In [1]:
def main(datasources, start_date, end_date):
    import dai
    import numpy as np
    import pandas as pd

    """
    F08: negative CAPM idiosyncratic skewness.

    The factor adapts Isc1 from the assignment PDF to daily BigAlpha output.
    We estimate each stock's rolling CAPM residual and compute the skewness of
    residuals over the recent window. Higher factor means lower idiosyncratic
    lottery-like skewness.
    """
    def rolling_idio_skew(group, window=20, min_periods=15):
        group = group.sort_values("date").copy()
        y = group["stock_return"]
        x = group["market_return"]

        mean_y = y.rolling(window, min_periods=min_periods).mean()
        mean_x = x.rolling(window, min_periods=min_periods).mean()
        cov_yx = (y * x).rolling(window, min_periods=min_periods).mean() - mean_y * mean_x
        var_x = (x * x).rolling(window, min_periods=min_periods).mean() - mean_x * mean_x

        beta = np.where(var_x.abs() > 1e-12, cov_yx / var_x, np.nan)
        alpha = mean_y - beta * mean_x
        residual = y - alpha - beta * x

        group["idio_skew"] = residual.rolling(window, min_periods=min_periods).skew()
        return group

    table_name = datasources["bar1m"]
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=90)).strftime("%Y-%m-%d %H:%M:%S")

    stock_daily = dai.query(
        f"""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            LAST(CAST(close AS DOUBLE) ORDER BY date) AS close
        FROM {table_name}
        GROUP BY date::DATE, instrument
        ORDER BY date, instrument
        """,
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()

    stock_daily["date"] = pd.to_datetime(stock_daily["date"]).dt.normalize()
    stock_daily["close"] = pd.to_numeric(stock_daily["close"], errors="coerce")
    stock_daily = stock_daily.sort_values(["instrument", "date"])
    stock_daily["stock_return"] = stock_daily.groupby("instrument")["close"].pct_change()

    try:
        market_daily = dai.query(
            """
            SELECT
                date,
                instrument,
                close
            FROM cn_stock_index_bar1d
            """,
            filters={
                "date": [query_start, end_date],
                "instrument": ["000852.SH"],
            },
            compression=True,
        ).df()
        market_daily["date"] = pd.to_datetime(market_daily["date"]).dt.normalize()
        market_daily["close"] = pd.to_numeric(market_daily["close"], errors="coerce")
        market_daily = market_daily.sort_values("date")
        market_daily["market_return"] = market_daily["close"].pct_change()
        market_daily = market_daily[["date", "market_return"]]
    except Exception:
        market_daily = (
            stock_daily.groupby("date", as_index=False)["stock_return"]
            .mean()
            .rename(columns={"stock_return": "market_return"})
        )

    work = stock_daily.merge(market_daily, on="date", how="inner")
    work = work.dropna(subset=["stock_return", "market_return"])

    work = (
        work.groupby("instrument", group_keys=False)
        .apply(rolling_idio_skew)
        .reset_index(drop=True)
    )

    work["factor"] = -work["idio_skew"]

    factor = work[["date", "instrument", "factor"]].copy()
    factor = factor[
        (factor["date"] >= pd.to_datetime(start_date).normalize())
        & (factor["date"] <= pd.to_datetime(end_date).normalize())
    ]
    factor["factor"] = pd.to_numeric(factor["factor"], errors="coerce")

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    return (
        pd.merge(factor, stk_pool, how="inner", on=["date", "instrument"])
        .dropna(subset=["factor"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )
